# Predicted vs. Actual Plot

This notebook uses `predicted_vs_actual.py` in the same folder to create a predicted-versus-actual scatter plot.

Audience:
- Students and researchers learning how to evaluate regression predictions.

Prerequisites:
- A CSV or Excel file with one Actual column and one Predicted column.
- Basic Python and pandas-style tabular data concepts.

Learning goals:
- Select the input file and the two columns used for evaluation.
- Interpret the 1:1 reference line and fitted regression line.
- Read R², RMSE, and MAE from the resulting plot.

## Outline

1. Load the plotting script and choose the input file
2. Inspect the available columns
3. Select Actual and Predicted columns
4. Create and inspect the plot
5. Common pitfalls and exercises

In [ ]:
# Run this only if the required packages are not installed.
# %pip install -r ../requirements.txt

from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
SCRIPT_PATH = NOTEBOOK_DIR / "predicted_vs_actual.py"
if not SCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {SCRIPT_PATH}. Open this notebook from the predicted-actual-plot folder."
    )

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from predicted_vs_actual import (
    calculate_metrics,
    extract_numeric_pairs,
    make_plot,
    read_table,
    resolve_column,
)

print(f"Using script: {SCRIPT_PATH}")

## Step 1 - Choose the input file

Set `filename` to a CSV, XLSX, or XLSM file. The supplied synthetic CSV is used by default so the notebook runs from top to bottom. Set it to `None` when you want to type a path interactively.

In [ ]:
# Replace this with your own file path when needed.
filename = "synthetic_actual_vs_predicted.csv"

# To enter a path when the cell runs, use:
# filename = input("Input file path: " ).strip()

headers, rows = read_table(filename)
print(f"Loaded {len(rows)} data rows from {filename}")
print("Available columns:")
for index, header in enumerate(headers, start=1):
    print(f"  {index}. {header}")

## Step 2 - Tell the script which columns to use

Enter column names from the list above. `resolve_column()` matches them case-insensitively (so `Actual` finds `actual`) and reports the available columns if a name is not found.

In [ ]:
actual_column = resolve_column(headers, "actual")
predicted_column = resolve_column(headers, "predicted")

actual, predicted, skipped_rows = extract_numeric_pairs(
    rows, actual_column, predicted_column
)
metrics = calculate_metrics(actual, predicted)

print(f"Using Actual column: {actual_column}")
print(f"Using Predicted column: {predicted_column}")
print(f"Complete pairs: {len(actual)}")
print(f"Skipped blank rows: {skipped_rows}")
metrics

## Step 3 - Create the plot

The x-axis is labelled **Actual values** and the y-axis **Predicted values**. The plot includes the dashed 1:1 identity line, the fitted regression line, and the R²/RMSE/MAE annotation.

In [ ]:
from IPython.display import Image, display

output_path = Path(filename).with_name(
    f"{Path(filename).stem}_predicted_vs_actual.png"
)

saved_path = make_plot(
    actual,
    predicted,
    metrics,
    actual_column,
    predicted_column,
    output_path,
)
print(f"PNG saved to: {saved_path}")

display(Image(filename=str(saved_path)))

## Common pitfalls and exercises

- The two selected columns must contain numeric values. Blank pairs are skipped; text values raise an error.
- The 1:1 line represents perfect prediction; it is not fitted to the data.
- Try `synthetic_actual_vs_predicted_r2_0.78.csv` and `synthetic_actual_vs_predicted_r2_0.60.csv` and compare how the spread around the 1:1 line changes R², RMSE, and MAE.
- Exercise: change `actual_column` and `predicted_column` to two valid numeric columns in your own file, then rerun the final two cells.